<a href="https://colab.research.google.com/github/werowe/HypatiaAcademy/blob/master/ml/golf_random_forest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Why Random Forests?  

Random Forests are powerful ensemble learning methods capable of handling both **classification** and **regression** tasks. They achieve this by constructing multiple decision trees during training and outputting the mode of the classes (for classification) or the mean prediction (for regression) of the individual trees.

Key characteristics that make them highly effective include:

*   **Reduces Overfitting**: By combining predictions from many trees, Random Forests significantly reduce the risk of overfitting, which occurs when a model performs exceptionally well on training data but poorly on unseen data.

*   **Handles High Dimensionality**: They can effectively manage datasets with a large number of features without requiring extensive feature selection.

*   **Provides Feature Importance**: Random Forests can quantify the importance of each feature in making predictions, offering valuable insights into the underlying data relationships.

*   **Robust to Noise**: They are less sensitive to noisy data or outliers, making them more reliable in real-world scenarios.

# Important Note on Synthetic Data

While real-world data is always preferred for building robust models.  But here we just make up some sample data since this is a teach example.  But what do we do about the label (Y)?  Obviously most golfers will golf in good weather, except the British where they don't often get good weather.  They play in the rain.  We can't interview 1,000 golfers to build a truly real world data set, so we make up some data.  But what to do about the label part of the feature/label data:  The decision whether to golf?  Obviously most people will golf when it's sunny, not too windy, and not too cold?  But we need some variation.

So we simulate the golf decision by adding a small amount of random noise to simulate read world conditions.


In [1]:
import pandas as pd
import numpy as np

# Set a random seed for reproducibility
np.random.seed(42)

# Generate sample data
data_size = 100

raining = np.random.choice([0, 1], size=data_size)  # 0 for No, 1 for Yes
windy = np.random.choice([0, 1], size=data_size)    # 0 for No, 1 for Yes
cold = np.random.choice([0, 1], size=data_size)       # 0 for No, 1 for Yes

# Create a simplified logic for 'should_golf'
# For example: If not raining AND not too windy AND not too cold, then golf.
# Otherwise, don't golf.
should_golf = ((raining == 0) & (windy == 0) & (cold == 0)) | \
              ((raining == 0) & (windy == 0) & (cold == 1) & (np.random.rand(data_size) > 0.3)) | \
              ((raining == 0) & (windy == 1) & (cold == 0) & (np.random.rand(data_size) > 0.6))

should_golf = should_golf.astype(int) # Convert boolean to 0 or 1

# Create a DataFrame
df = pd.DataFrame({
    'Raining': raining,
    'Windy': windy,
    'Cold': cold,
    'Golf': should_golf
})

print(df.head())
print(f"\nValue counts for 'Golf':\n{df['Golf'].value_counts()}")

   Raining  Windy  Cold  Golf
0        0      0     0     1
1        1      1     1     0
2        0      1     0     1
3        0      1     0     0
4        0      1     1     0

Value counts for 'Golf':
Golf
0    75
1    25
Name: count, dtype: int64


# Explanation of Random Forest Principles: Bagging and Feature Sampling

Random Forests derive their power and robustness from two core ensemble techniques: **Bagging (Bootstrap Aggregating)** and **Feature Sampling (Random Subspace Method)**.

## Bagging (Bootstrap Aggregating)

Bagging is a powerful ensemble meta-algorithm that builds multiple models (typically of the same type, like decision trees) on different subsets of the training data and then combines their predictions. For Random Forests, this process involves:

1.  **Bootstrapping**: Instead of training each decision tree on the entire training dataset, a Random Forest generates multiple, diverse subsets of the data by **sampling with replacement**. This means each bootstrap sample is roughly the same size as the original dataset, but some data points may appear multiple times, while others might be excluded. This creates varied training sets for each tree.

2.  **Parallel Training**: Each decision tree within the forest is trained independently on its unique bootstrap sample. This parallelism is key to efficiency and diversity.

3.  **Aggregation**: After all trees are trained, their individual predictions are combined. For **classification** tasks (like our 'Golf' prediction), the final prediction is determined by a **majority vote** among all trees. For **regression** tasks, the final prediction is typically the average of all individual tree predictions.

This bootstrapped training helps reduce the variance of the overall model and mitigate overfitting, as the aggregated decisions are more stable and reliable than any single tree's decision.

## Feature Sampling (Random Subspace Method)

Beyond data bagging, Random Forests introduce another layer of randomness by sampling features when building each individual decision tree:

1.  **Random Subset of Features**: When a decision tree is being constructed and needs to determine the best split at a given node, it does not consider all available features. Instead, it randomly selects a **subset of features** to evaluate. For classification problems, a common rule of thumb is to consider `sqrt(total_number_of_features)` at each split. For example, with 9 features, it might only consider 3 features (`sqrt(9)`) for a split.

2.  **Reduced Correlation**: This random feature selection further decorrelates the individual trees in the forest. If all trees were allowed to consider all features at every split, they might all converge on the same strongest predictors, leading to highly correlated trees and less benefit from ensembling.

By combining both **bagging** (sampling data) and **feature sampling** (sampling features), Random Forests create a diverse ensemble of trees that are robust, less prone to overfitting, and often achieve superior predictive accuracy compared to simpler machine learning models.

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score


# Define features (X) and target (y)
X = df[['Raining', 'Windy', 'Cold']]
y = df['Golf']

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")


X_train shape: (70, 3)
X_test shape: (30, 3)
y_train shape: (70,)
y_test shape: (30,)


# Random Forest Classifier Parameters

When initializing the `RandomForestClassifier`, key parameters include:

*   `n_estimators`: This crucial parameter specifies the number of individual decision trees that will be built in the forest. A larger number typically leads to a more stable and accurate model, but also increases computational cost. Each of these trees is trained on a unique bootstrap sample of the data.

*   `random_state`: Setting a `random_state` ensures that the random processes (like data bootstrapping and feature sampling) are reproducible. This means you will get the exact same results every time you run the code with the same `random_state`.

By default, `RandomForestClassifier` uses `bootstrap=True` for bagging and `max_features='sqrt'` for feature sampling (meaning it considers the square root of the total number of features at each split). These defaults are generally good starting points.



In [3]:
from sklearn.ensemble import RandomForestClassifier


rf_classifier = RandomForestClassifier(n_estimators=100, random_state=42)

print("RandomForestClassifier initialized:")
print(rf_classifier)

RandomForestClassifier initialized:
RandomForestClassifier(random_state=42)


In [4]:
# Train the Random Forest Classifier
rf_classifier.fit(X_train, y_train)

print("RandomForestClassifier trained successfully.")

RandomForestClassifier trained successfully.


In [5]:
from sklearn.ensemble import RandomForestClassifier

# Make predictions on the test set
y_pred = rf_classifier.predict(X_test)

print("Predictions on the test set have been made.")

Predictions on the test set have been made.


### Model Evaluation (`.predict()` and `accuracy_score()`)

After training our Random Forest model, the next critical step is to evaluate its performance. We use the `predict()` method to generate predictions (`y_pred`) on the `X_test` dataset, which the model has not seen before. These predictions are then compared against the actual true labels (`y_test`) to assess how well our model generalizes to new data.

For classification tasks, several metrics can be used for evaluation, including accuracy, precision, recall, F1-score, and ROC AUC. In this example, we will calculate the **accuracy score**, which provides a straightforward measure of the proportion of correctly classified instances.

In [6]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, y_pred)

print(f"Model Accuracy on the test set: {accuracy:.2f}")

Model Accuracy on the test set: 0.87
